Import Libraries


In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import streamlit as st

Load Dataset

In [17]:
df = pd.read_csv("reduced_data2.csv")   
df.head()

,,date,user,pc,to,cc,bcc,from,size,attachments,content,day_of_week,email_count,insider_threat
0,{M7B0-V8OW91OT-6342OLKY},3/9/2010 17:14,GCR0554,PC-0191,Philip.Travis.Hartman@dtaa.com;Britanni.Marah....,Nissim.Rafael.Delacruz@dtaa.com,NaN,Grace.Charde.Roth@dtaa.com,29647,0,nearest sufficient primary john heavy century ...,1,1,0
1,{B8C5-Q8IL02JQ-9873ZEMP},2/19/2010 10:58,VMJ0147,PC-3873,Libby.Rosalyn.Richard@dtaa.com,NaN,NaN,Vera.Mollie.Jenkins@dtaa.com,41499,0,pursuing merit older whether their parts josh ...,4,1,0
2,{U6L9-R3CJ21VI-4023IPPV},3/9/2010 16:32,BHC0665,PC-6706,Nora.Regina.Oconnor@dtaa.com,Akeem.William.Mayer@dtaa.com;Blaine.Helen.Cerv...,NaN,Blaine.Helen.Cervantes@dtaa.com,30105,4,event 17 needs spending deals america door sti...,1,1,1
3,{N8S7-K7KM81XN-5165EZTQ},2/26/2010 7:46,KCF0045,PC-8845,Atkinson-Cyrus@charter.net,Megan.S.Hess@aol.com,NaN,Frank-Kendall@netzero.com,36564,0,upward has arctic temperature amount have more...,4,1,1
4,{P4U9-M0VU92US-9183HADC},2/4/2010 15:12,RHY0079,PC-1896,Bell.Jorden.Martinez@dtaa.com,NaN,NaN,Rose.Hanae.Yates@dtaa.com,23470,0,remains rabbit group everything ohs over fault...,3,1,1


Data Preprocessing(Handle Missing Values)

In [18]:
df.isnull().sum()

                     0
date                 0
user                 0
pc                   0
to                   0
cc                1260
bcc               1673
from                 0
size                 0
attachments          0
content              0
day_of_week          0
email_count          0
insider_threat       0
dtype: int64

In [19]:
df = df.dropna()

In [20]:
print(df.columns)

Index([' ', 'date', 'user', 'pc', 'to', 'cc', 'bcc', 'from', 'size',
       'attachments', 'content', 'day_of_week', 'email_count',
       'insider_threat'],
      dtype='str')


Feature Engineering

In [21]:
df['email_count'] = df.groupby('from')['from'].transform('count')
#replaces missing attachments with 0
df['attachments'] = df['attachments'].fillna(0)

Creation of Target Variable

In [22]:
#Apply conditions inside the loc function to set insider_threat to 1 for high-risk emails
df['insider_threat'] = 0
df.loc[
    (df['email_count'] > 30) |    
    (df['attachments'] > 5) |    
    (df['day_of_week'] >= 4),    
    'insider_threat'
] = 1

Define Features and Target

In [23]:
#To standardize the features, we can use StandardScaler from sklearn.preprocessing
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
#input elements for the model
X = df[['email_count', 'attachments', 'day_of_week']]
#target variable
y = df['insider_threat']
X = scaler.fit_transform(X)

Train Test Split

In [24]:
#The model is trained on 80% of the data and evaluated on 
#the remaining 20% to check how well it performs on unseen data.
from sklearn.model_selection import train_test_split
xtrain, xtest, ytrain, ytest = train_test_split(
    X, y, test_size=0.2, random_state=42
)

Train Model - LOGISTIC REGRESSION

In [25]:
#Trained a Logistic Regression model to classify employees as normal or insider threats.”
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(max_iter=1000)
lr.fit(xtrain, ytrain)
ypred_lr = lr.predict(xtest)

Train Model - NAIVE BAYES

In [26]:
#Training another model for comparison
from sklearn.naive_bayes import GaussianNB
nb = GaussianNB()
nb.fit(xtrain, ytrain)
# Naive bayes
ypred_nb = nb.predict(xtest)
# Logistic Regression
ypred_lr = lr.predict(xtest)
print("LR:", ypred_lr)
print("NB:", ypred_nb)

LR: [0 1 0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 1 0 1 1 0 0 0 0 0]
NB: [0 1 0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 1 0 1 1 0 1 0 0 0]


In [27]:
# 1. Train model
lr = LogisticRegression(max_iter=1000)
lr.fit(xtrain, ytrain)
# 2. Predictions
df['prediction'] = lr.predict(X)
df['risk_score'] = lr.predict_proba(X)[:,1]
# 3. Filter high-risk users
high_risk_users = df[df['prediction'] == 1]
high_risk_list = high_risk_users['from'].unique()
# 4. Display in Streamlit
st.subheader("🚨 High Risk Employees")
if len(high_risk_list) > 0:
    st.write(list(high_risk_list))
else:
    st.success("No high risk users detected")

2026-06-16 09:05:21.650 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-16 09:05:22.040 
  command:

    streamlit run C:\Users\SHREEYA\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-06-16 09:05:22.040 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-16 09:05:22.040 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-16 09:05:22.055 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-16 09:05:22.055 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-16 09:05:22.055 Thread 'MainThread': missing ScriptRunContext! This

Evaluation Metrics

In [28]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
print("Logistic Regression:")
print("Accuracy:", accuracy_score(ytest, ypred_lr))#overall correctness
print("Precision:", precision_score(ytest, ypred_lr))#how many predicted threats are actually threats
print("Recall:", recall_score(ytest, ypred_lr))#how many real threats were found
print("F1 Score:", f1_score(ytest, ypred_lr))#Balanced measure of precision and recall
print("Confusion Matrix:\n", confusion_matrix(ytest, ypred_lr))#true positives, false positives, true negatives, false negatives

Logistic Regression:
Accuracy: 0.9615384615384616
Precision: 1.0
Recall: 0.8571428571428571
F1 Score: 0.9230769230769231
Confusion Matrix:
 [[19  0]
 [ 1  6]]


Naive Bayes Result

In [29]:
print("\nNaive Bayes:")
print("Accuracy:", accuracy_score(ytest, ypred_nb))
print("Precision:", precision_score(ytest, ypred_nb))
print("Recall:", recall_score(ytest, ypred_nb))
print("F1 Score:", f1_score(ytest, ypred_nb))
import seaborn as sns
import matplotlib.pyplot as plt
cm = confusion_matrix(ytest, ypred_lr)
plt.figure()
sns.heatmap(cm, annot=True, fmt='d')
plt.title("Confusion Matrix - Logistic Regression")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


Naive Bayes:
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1 Score: 1.0


C:\Users\SHREEYA\AppData\Local\Temp\ipykernel_17636\983339196.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [30]:
import matplotlib.pyplot as plt
df['insider_threat'].value_counts().plot(kind='bar')
plt.title("Normal vs Threat")
plt.show()

C:\Users\SHREEYA\AppData\Local\Temp\ipykernel_17636\4067528839.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [31]:
df['insider_threat'].value_counts()

insider_threat
0    101
1     27
Name: count, dtype: int64

In [32]:
comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "Naive Bayes"],
    "Accuracy": [
        accuracy_score(ytest, ypred_lr),
        accuracy_score(ytest, ypred_nb)
    ],
    "Precision": [
        precision_score(ytest, ypred_lr, zero_division=0),
        precision_score(ytest, ypred_nb, zero_division=0)
    ],
    "Recall": [
        recall_score(ytest, ypred_lr, zero_division=0),
        recall_score(ytest, ypred_nb, zero_division=0)
    ],
    "F1 Score": [
        f1_score(ytest, ypred_lr, zero_division=0),
        f1_score(ytest, ypred_nb, zero_division=0)
    ]
})
print("\nModel Comparison:")
print(comparison)


Model Comparison:
                 Model  Accuracy  Precision    Recall  F1 Score
0  Logistic Regression  0.961538        1.0  0.857143  0.923077
1          Naive Bayes  1.000000        1.0  1.000000  1.000000


In [33]:
# Probability-based risk score
prob = lr.predict_proba(xtest)[:, 1]
print("\nSample Risk Scores:")
print(prob[:5])


Sample Risk Scores:
[0.31771278 0.97111385 0.01137777 0.04288971 0.11553182]


In [34]:
df = pd.read_csv("reduced_data2.csv")

In [35]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# Features and target
X = df[['email_count', 'attachments', 'day_of_week']]
y = df['insider_threat']

# Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "KNN": KNeighborsClassifier(),
    "SVM": SVC()
}

# Train and compare
for name, model in models.items():
    model.fit(X_scaled, y)
    pred = model.predict(X_scaled)
    acc = accuracy_score(y, pred)
    print(f"{name}: {acc:.4f}")

Logistic Regression: 0.9210
Decision Tree: 0.9550
Random Forest: 0.9550
KNN: 0.9535
SVM: 0.9550


In [14]:
print(data.dtypes)

                    str
date                str
user                str
pc                  str
to                  str
cc                  str
bcc                 str
from                str
size              int64
attachments       int64
content             str
day_of_week       int64
email_count       int64
insider_threat    int64
dtype: object


In [36]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(random_state=42)
dt.fit(xtrain, ytrain)

ypred_dt = dt.predict(xtest)

In [37]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(xtrain, ytrain)

ypred_rf = rf.predict(xtest)

In [38]:
from sklearn.svm import SVC

svm = SVC(kernel='rbf')
svm.fit(xtrain, ytrain)

ypred_svm = svm.predict(xtest)

In [39]:
from sklearn.metrics import accuracy_score

print("Logistic Regression:", accuracy_score(ytest, ypred_lr))
print("Decision Tree:", accuracy_score(ytest, ypred_dt))
print("Random Forest:", accuracy_score(ytest, ypred_rf))
print("SVM:", accuracy_score(ytest, ypred_svm))

Logistic Regression: 0.9615384615384616
Decision Tree: 1.0
Random Forest: 1.0
SVM: 0.8846153846153846
